# 02. 소둔로 정비와 부품교체 연결

`P1-CR1-CAL01` 정비이력을 작업번호(`WO_NO`), 부품코드(`PART_CD`)로 연결함. 교육용 재현 자료이며 정비 기록만으로 센서 고장을 확정하지 않음.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display

# 저장소 루트, notebooks 폴더, 개인 노트북 폴더에서 같은 파일을 읽음.
if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')

TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(DATA_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(DATA_DIR, 'T-CR1-WLD01_용접.csv')  # 현재 분석에서는 용접 파일을 읽지 않음.
ENC = 'utf-8-sig'

FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark', '02_maintenance_parts')
OUTPUT_DIR = os.path.join(DATA_DIR, 'processed', 'analysis', 'maintenance', 'v1')

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

# 설치된 한글 폰트를 사용함.
font_names = []

for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)

for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if font_name in font_names:
        rcParams['font.family'] = font_name
        break
else:
    print('경고: 사용할 수 있는 한글 폰트를 찾지 못했음. 그래프의 한글이 깨질 수 있음.')

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110

In [ ]:
# 이 노트북에서만 쓰는 추가 설정
import matplotlib.dates as mdates
from IPython.display import display

CAL_TAG = "P1-CR1-CAL01"  # 연속소둔로 설비 태그

# 정비유형별 색 - 모든 그래프에서 같은 색 = 같은 정비유형
MNT_ORDER = ["TBM", "BM", "CBM"]
MNT_COLORS = {"TBM": "#2a78d6", "BM": "#eb6834", "CBM": "#1baf7a"}

# 그래프 기본 모양: 위, 오른쪽 테두리 없애고 옅은 격자
rcParams["axes.spines.top"] = False
rcParams["axes.spines.right"] = False
rcParams["axes.grid"] = True
rcParams["grid.color"] = "#e1e0d9"
rcParams["grid.linewidth"] = 0.6

# 그림은 이 노트북 전용 폴더에 저장함.

## 1. 데이터 불러오기

원본 날짜를 날짜형으로 변환함.


In [ ]:
# CSV를 직접 읽고, 계산에 사용하는 날짜 컬럼을 날짜형으로 바꿈.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 읽은 원본 데이터프레임을 이름별로 보관함.
sources = {
    'temp': temp,
    'event': event,
    'repair': repair,
    'eqp': eqp,
    'part': part,
    'maint': maint,
    'partchg': partchg,
}


In [ ]:
# 정비와 부품교체를 연결함.
# 소둔로 정비에 부품 정보를 연결함.

maintenance_base = maint.loc[maint["EQP_TAG"].eq("P1-CR1-CAL01")].copy()

replacement_rows = partchg.loc[partchg["WO_NO"].isin(maintenance_base["WO_NO"])].copy()

replacement_rows = replacement_rows.merge(
    part, on="PART_CD", how="left", validate="many_to_one", indicator=True,
)

replacement_rows = replacement_rows.drop(columns="_merge")


In [ ]:
# 부품 여러 개가 달린 정비도 정비 한 건당 한 행으로 요약함.
replacement_summary = replacement_rows.groupby("WO_NO", as_index=False).agg(
    PART_RECORD_COUNT=("PART_CD", "size"),
    PART_QTY_TOTAL=("QTY", "sum"),
)

replacement_summary["PART_CODES"] = ""
replacement_summary["PART_NAMES"] = ""
replacement_summary["PART_CHANGE_REASONS"] = ""

for replacement_wo, replacement_group in replacement_rows.groupby("WO_NO"):
    replacement_hit = replacement_summary["WO_NO"].eq(replacement_wo)

    replacement_summary.loc[replacement_hit, "PART_CODES"] = " | ".join(replacement_group["PART_CD"].astype(str))
    replacement_summary.loc[replacement_hit, "PART_NAMES"] = " | ".join(replacement_group["PART_NM"].astype(str))
    replacement_summary.loc[replacement_hit, "PART_CHANGE_REASONS"] = " | ".join(replacement_group["CHG_RSN"].astype(str))

replacement_summary = replacement_summary[[
    "WO_NO", "PART_RECORD_COUNT", "PART_CODES", "PART_NAMES", "PART_QTY_TOTAL", "PART_CHANGE_REASONS",
]]

# left join으로 교체 부품이 없는 BM, TBM, CBM도 남김.
maintenance_base = maintenance_base.merge(
    replacement_summary, on="WO_NO", how="left", validate="one_to_one",
)

maintenance_base["HAS_PART_RECORD"] = maintenance_base["PART_RECORD_COUNT"].notna()

for replacement_column in ["PART_RECORD_COUNT", "PART_QTY_TOTAL"]:
    maintenance_base[replacement_column] = maintenance_base[replacement_column].fillna(0).astype(int)

for replacement_column in ["PART_CODES", "PART_NAMES", "PART_CHANGE_REASONS"]:
    maintenance_base[replacement_column] = maintenance_base[replacement_column].fillna("")



In [ ]:
# 작업시간과 실제 정지시간 기록을 구분함. 정지 위치는 역산하지 않음.
maintenance_base["WORK_MIN"] = (maintenance_base["END_DT"] - maintenance_base["STRT_DT"]).dt.total_seconds() / 60

maintenance_base["DOWN_RECORD_STATUS"] = "정지시간 미기록"
maintenance_base.loc[maintenance_base["DOWN_MIN"].eq(0), "DOWN_RECORD_STATUS"] = "해당 정비의 정지시간 0 기록"
maintenance_base.loc[maintenance_base["DOWN_MIN"].gt(0), "DOWN_RECORD_STATUS"] = "정지시간 기록 있음; 시점 미확정"

maintenance_base["DOWN_TO_WORK_RATIO"] = maintenance_base["DOWN_MIN"].div(
    maintenance_base["WORK_MIN"].where(maintenance_base["WORK_MIN"].gt(0))
)

maintenance_base["DOWN_EXCEEDS_WORK"] = maintenance_base["DOWN_MIN"].astype("Float64").gt(
    maintenance_base["WORK_MIN"].astype("Float64")
)

maintenance_base["STOP_INTERVAL_KNOWN"] = False

maintenance_events = maintenance_base.sort_values(["DETC_DT", "WO_NO"]).reset_index(drop=True)

# 부품별 상세표는 실제 교체 기록이 있는 정비만 한 부품당 한 행으로 남김.
replacement_details = maint.loc[maint["EQP_TAG"].eq("P1-CR1-CAL01")].merge(
    replacement_rows, on="WO_NO", how="inner", validate="one_to_many",
)

part_details = replacement_details.sort_values(["STRT_DT", "WO_NO", "PART_CD"]).reset_index(drop=True)

print('연속소둔로 정비:', len(maintenance_events), '건 / 교체 부품 상세:', len(part_details), '행')


In [ ]:
# CR1 정기수리 일정 - 이후 그래프에서 회색으로 칠할 기간
cr1_repair = repair[repair["PLANT_CD"] == "CR1"]

display(cr1_repair)

## 2. 소둔로 정비이력

| 컬럼 | 계산과 의미 |
| --- | --- |
| `대기시간` | 감지(`DETC_DT`)부터 작업 착수(`STRT_DT`)까지 |
| `작업시간` | 작업 착수부터 완료(`END_DT`)까지 |

사전과 달리 계획정비에도 `DETC_DT`가 있어, 이 값을 모두 이상 감지 시각으로 해석하지 않음.


In [ ]:
cal = maintenance_events.copy()  # 부품이 없어도 남긴 정비 1건당 1행
cal = cal.sort_values("DETC_DT").reset_index(drop=True)

# 차이값을 시간 단위 숫자로 바꾸기
cal["대기시간"] = (cal["STRT_DT"] - cal["DETC_DT"]).dt.total_seconds() / 3600
cal["작업시간"] = (cal["END_DT"] - cal["STRT_DT"]).dt.total_seconds() / 3600

# FAIL_MODE 빈칸은 '고장이 없는 계획정비'일 수 있어서 지우지 않고 (공란)으로 채움
cal["FAIL_MODE"] = cal["FAIL_MODE"].fillna("(공란)")

# REMARK 앞뒤 공백 정리 ("  정기수리 부속작업" 처럼 앞에 공백이 붙은 값이 있음)
cal["REMARK"] = cal["REMARK"].str.strip()

print("전체 정비:", len(maint), "건 → 연속소둔로:", len(cal), "건")
display(cal.head())

### 2-1. 정비 44건의 종류

계획정비(`TBM`), 고장정비(`BM`), 상태를 보고 하는 정비(`CBM`)를 나누어 비교함.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (1) 정비유형
counts = cal["MNT_TYPE"].value_counts().reindex(MNT_ORDER)
maintenance_colors = []

for maintenance_type in MNT_ORDER:
    maintenance_colors.append(MNT_COLORS[maintenance_type])

bars = axes[0].bar(counts.index, counts.values, color=maintenance_colors)
axes[0].bar_label(bars)  # 막대 위에 건수 표시
axes[0].set_title("정비유형 (MNT_TYPE)")

# (2) 고장모드
counts = cal["FAIL_MODE"].value_counts()
bars = axes[1].bar(counts.index, counts.values, color="#8c8c8c")
axes[1].bar_label(bars)
axes[1].set_title("고장모드 (FAIL_MODE)")

# (3) 부품교체 여부
counts = cal["PART_CHG"].value_counts()
bars = axes[2].bar(counts.index, counts.values, color="#8c8c8c")
axes[2].bar_label(bars)
axes[2].set_title("부품교체 여부 (PART_CHG)")

for ax in axes:
    ax.set_ylabel("건수")

fig.suptitle("연속소둔로 정비 44건 구성", y=1.04)

plt.savefig(os.path.join(FIG_DIR, "02_mnt_counts.png"), bbox_inches="tight")

plt.show()

정비유형별 고장모드와 부품교체 여부를 비교함.


In [ ]:
# 정비유형 × 고장모드 교차표 (칸 안의 숫자 = 건수)
cross = pd.crosstab(cal["MNT_TYPE"], cal["FAIL_MODE"]).reindex(MNT_ORDER)

display(cross)

In [ ]:
# 유형별 교체 유무도 확인함. BM만 따로 뽑지 않음.
display(pd.crosstab(cal['MNT_TYPE'], cal['PART_CHG']).reindex(MNT_ORDER))



계획정비 22건, 고장정비 12건, 상태를 보고 한 정비 10건이며, 이 중 부품교체는 8건임. 고장모드 공란 24건 중 20건은 계획정비, 나머지 4건은 원인 미상 등으로 남겨 둠.

고장모드 코드는 전기(`ELE`), 마모(`WER`), 정렬불량(`MAL`), 기타(`ETC`)임. 교육용 설정값이므로 실제 고장 통계로 해석하지 않음.


### 2-2. 감지부터 작업 착수까지

점 하나는 정비 1건이며 색은 정비유형임. 세 그림의 세로축은 같은 로그 눈금(`1 → 10 → 100시간`)임.


In [ ]:
np.random.seed(0)  # 점을 흩뿌리는 위치가 실행할 때마다 같도록 고정

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

for ax, col in zip(axes, ["MNT_TYPE", "FAIL_MODE", "PART_CHG"]):
    if col == "MNT_TYPE":
        groups = MNT_ORDER
    else:
        groups = sorted(cal[col].unique())

    # 상자그림: 가운데 선이 중앙값
    data = []

    for group_name in groups:
        data.append(cal.loc[cal[col] == group_name, "대기시간"])

    ax.boxplot(data, tick_labels=groups, showfliers=False, widths=0.5)

    # 점: 정비 1건, 색 = 정비유형 (겹치지 않게 옆으로 살짝 흩뿌림)
    for i, g in enumerate(groups, start=1):
        sub = cal[cal[col] == g]
        x = i + np.random.uniform(-0.15, 0.15, len(sub))
        ax.scatter(x, sub["대기시간"], c=sub["MNT_TYPE"].map(MNT_COLORS), s=35, edgecolor="white", zorder=3)

    ax.set_yscale("log")
    ax.set_yticks([0.01, 0.1, 1, 10, 100], ["0.01", "0.1", "1", "10", "100"])  # 눈금을 보통 숫자로
    ax.set_title((str(col) + '별 대기시간'))

axes[0].set_ylabel("이상탐지 → 작업착수 (시간, 로그 눈금)")

for t in MNT_ORDER:  # 범례만 만들기 위해 데이터 없는 빈 점을 찍음
    axes[2].scatter([], [], c=MNT_COLORS[t], label=t)

axes[2].legend(title="정비유형", loc="upper left", bbox_to_anchor=(1.0, 1.0))

plt.savefig(os.path.join(FIG_DIR, "02_wait_hours_by_group.png"), bbox_inches="tight")

plt.show()

In [ ]:
# 숫자로도 확인: 건수 / 중앙값 / 최댓값 (단위: 시간)
display(cal.groupby("MNT_TYPE")["대기시간"].agg(["count", "median", "max"]).reindex(MNT_ORDER).round(2))
display(cal.groupby("PART_CHG")["대기시간"].agg(["count", "median", "max"]).round(2))

계획정비와 고장정비는 수 시간 안에 착수했고, 상태를 보고 한 정비는 대기시간 중앙값 약 117시간, 최대 약 216시간이었음. 고장모드와 부품교체 여부별 차이에도 정비유형의 구성이 영향을 줌.

상태를 보고 한 정비 중 설비가 보호장치로 멈췄다는 기록 2건도 약 68시간, 110시간 뒤에 착수했고, 2건은 하루 안에 착수했음. 따라서 긴 대기시간을 계획된 여유나 모델이 미리 경고할 수 있는 시간으로 단정하지 않음.


### 2-3. 작업시간과 정지시간

[정비 데이터사전](https://app.notion.com/p/26be609a45d0838fb0c88106c95f4c02)의 `DOWN_MIN`은 실제 정지시간(분)이며 작업시간과 다르고, 0도 유효함. 정지 시작과 끝은 기록되지 않았으므로 작업 시각에 정지시간을 더하거나 빼서 임의의 정지 구간을 만들지 않음.


In [ ]:
downtime_summary = maintenance_events.groupby('MNT_TYPE').agg(
    정비건수=('WO_NO', 'size'),
    정지분중앙값=('DOWN_MIN', 'median'),
    작업분중앙값=('WORK_MIN', 'median'),
).reindex(['BM', 'TBM', 'CBM'])
downtime_summary['정지양수'] = maintenance_events['DOWN_MIN'].gt(0).groupby(maintenance_events['MNT_TYPE']).sum()
downtime_summary['정지0분'] = maintenance_events['DOWN_MIN'].eq(0).groupby(maintenance_events['MNT_TYPE']).sum()
downtime_summary['정지결측'] = maintenance_events['DOWN_MIN'].isna().groupby(maintenance_events['MNT_TYPE']).sum()

downtime_summary = downtime_summary[[
    '정비건수', '정지양수', '정지0분', '정지결측', '정지분중앙값', '작업분중앙값',
]]

for name, mask in {
    '작업시간과같음': maintenance_events.DOWN_MIN.eq(maintenance_events.WORK_MIN),
    '작업시간보다짧음': maintenance_events.DOWN_MIN.lt(maintenance_events.WORK_MIN),
    '작업시간보다김': maintenance_events.DOWN_MIN.gt(maintenance_events.WORK_MIN),
}.items():
    downtime_summary[name] = mask.groupby(maintenance_events.MNT_TYPE).sum().reindex(downtime_summary.index)

display(downtime_summary)
display(maintenance_events.loc[maintenance_events.DOWN_MIN.eq(0), [
    'WO_NO', 'MNT_TYPE', 'STRT_DT', 'END_DT', 'WORK_MIN', 'DOWN_MIN', 'REMARK', 'PART_CHG'
]])

아래 행수는 작업 착수 이상, 완료 미만에 기록된 센서 행수이며 정지 중 관측 행수는 아님. 작업별 `DOWN_MIN` 합계와 정기수리 계획도 설비 전체의 실제 정지 구간을 뜻하지 않음.


In [ ]:
# LOT의 실제 첫 관측과 마지막 관측을 구함.

lot_bounds = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    ROWS=("MEAS_DT", "size"),
)

# CR1 강종변경 중 LOT 첫 관측과 시각이 정확히 같은 기록만 연결함.
lot_grade_changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_bounds = lot_bounds.merge(
    lot_grade_changes[["EVT_DT", "AFT_VAL"]],
    left_on="START", right_on="EVT_DT", how="left", validate="many_to_one",
)

lot_bounds = lot_bounds.rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")



In [ ]:
# LOT별 관측 기간과 강종을 정리함.

lot_bounds["DURATION_H"] = (lot_bounds["END"] - lot_bounds["START"]).dt.total_seconds() / 3600

lot_context = lot_bounds[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]

lot_context = lot_context.sort_values("START").reset_index(drop=True)


In [ ]:
# 사용자 제외 구간을 표시한 뒤 작업시간과 비교함.
# 사용자 지정 네 경계만 제외함. 경계값과 비교 방향을 그대로 표시함.
exclusion_rules = [
    (240058, "2024-06-09 08:00:00", "at_or_after", "사용자 지정: 노 내림 구간 제외"),
    (240061, "2024-06-15 02:30:00", "before", "사용자 지정: 승온 구간 제외"),
    (240114, "2024-11-04 20:00:00", "at_or_after", "사용자 지정: 정기수리 전 하강 구간 제외"),
    (240117, "2024-11-13 09:00:00", "before", "사용자 지정: 정기수리 후 승온 구간 제외"),
]

marked_temp = temp.copy()

marked_temp["EXCLUDED"] = False
marked_temp["EXCLUSION_REASON"] = ""
exclusion_rows = []

for exclusion_lot, exclusion_boundary, exclusion_direction, exclusion_reason in exclusion_rules:
    exclusion_boundary = pd.Timestamp(exclusion_boundary)

    if exclusion_direction == "at_or_after":
        exclusion_time = marked_temp["MEAS_DT"].ge(exclusion_boundary)
    else:
        exclusion_time = marked_temp["MEAS_DT"].lt(exclusion_boundary)

    exclusion_hit = marked_temp["LOT_NO"].eq(exclusion_lot) & exclusion_time

    marked_temp.loc[exclusion_hit, "EXCLUDED"] = True
    marked_temp.loc[exclusion_hit, "EXCLUSION_REASON"] = exclusion_reason

    exclusion_rows.append({
        "LOT_NO": exclusion_lot, "boundary": exclusion_boundary,
        "direction": exclusion_direction, "excluded_rows": int(exclusion_hit.sum()),
    })

_ = pd.DataFrame(exclusion_rows)


In [ ]:
# 정비 작업시간과 실제 관측의 겹침을 계산함.
overlap_records = []

for work in maintenance_events.itertuples(index=False):
    # 작업 시간과 실제 관측의 겹침만 계산함. 정지 시간을 이 위치로 배정하지 않음.
    hit = marked_temp.loc[marked_temp.MEAS_DT.ge(work.STRT_DT) & marked_temp.MEAS_DT.lt(work.END_DT)]

    for lot_no, observed in hit.groupby('LOT_NO'):
        overlap_records.append({
            'LOT_NO': lot_no, 'WO_NO': work.WO_NO, 'MNT_TYPE': work.MNT_TYPE,
            'STRT_DT': work.STRT_DT, 'END_DT': work.END_DT,
            'WORK_MIN': work.WORK_MIN, 'DOWN_MIN': work.DOWN_MIN,
            'DOWN_RECORD_STATUS': work.DOWN_RECORD_STATUS, 'STOP_INTERVAL_KNOWN': work.STOP_INTERVAL_KNOWN,
            'OBSERVED_START': observed.MEAS_DT.min(), 'OBSERVED_END': observed.MEAS_DT.max(),
            'OBSERVED_WORK_OVERLAP_ROWS': len(observed),
            'RETAINED_WORK_OVERLAP_ROWS': int((~observed.EXCLUDED).sum()),
            'PART_CHG': work.PART_CHG, 'PART_NAMES': work.PART_NAMES,
        })

work_observation_overlap = pd.DataFrame(overlap_records)

display(work_observation_overlap)

downtime_summary.to_csv(os.path.join(OUTPUT_DIR, '02_downtime_summary.csv'), encoding='utf-8-sig')
work_observation_overlap.to_csv(os.path.join(OUTPUT_DIR, '02_work_observation_overlap.csv'), index=False, encoding='utf-8-sig')


### 2-4. 정비와 정기수리가 겹친 시기

6월, 11월 정기수리 중 수행한 작업과 그 밖의 정비를 나누어 봄.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.2))

# 회색 = CR1 정기수리 (종료일 당일까지 포함하려고 +1일)
for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

# 정비유형별로 한 줄씩 점 찍기
for i, t in enumerate(MNT_ORDER):
    sub = cal[cal["MNT_TYPE"] == t]
    ax.scatter(sub["DETC_DT"], [i] * len(sub), color=MNT_COLORS[t], s=45, edgecolor="white", zorder=3, label=t)

ax.set_yticks([0, 1, 2], MNT_ORDER)
ax.set_ylim(-0.7, 2.7)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 정비 탐지 시점 (회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))

plt.savefig(os.path.join(FIG_DIR, "02_mnt_timeline.png"), bbox_inches="tight")

plt.show()

In [ ]:
# 정기수리 기간(종료일 포함) 안에 탐지된 정비
cal["정기수리중"] = False

for _, r in cr1_repair.iterrows():
    inside = (cal["DETC_DT"] >= r["STRT_DT"]) & (cal["DETC_DT"] < r["END_DT"] + pd.Timedelta(days=1))
    cal.loc[inside, "정기수리중"] = True

print("정기수리 기간 안에 탐지된 정비:", cal["정기수리중"].sum(), "건")
display(cal.loc[cal["정기수리중"], ["WO_NO", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "REMARK"]])

회색 정기수리 기간에 감지된 11건은 모두 계획정비임. 9건은 수리 일정 관련 기록, 2건은 결함 기록(`WO-2024-00931` 마모량 초과, `WO-2024-00916` O링 손상)이었음.

그림과 `정기수리중` 표시는 종료일을 포함한 기준임. 종료일 포함 여부 비교는 [01](01_data_alignment.ipynb)에 있음.


## 3. 작업번호로 부품교체 연결

부품교체이력에는 날짜가 없으므로 `WO_NO`로 정비이력에 연결함.


In [ ]:
wo_y = set(maint.loc[maint["PART_CHG"] == "Y", "WO_NO"])  # 정비이력: 부품교체 = Y 인 작업지시

wo_chg = set(partchg["WO_NO"])                            # 부품교체이력에 등장한 작업지시

print("정비이력 PART_CHG=Y:", len(wo_y), "건 / 부품교체이력 WO_NO:", len(wo_chg), "개")
print("Y인데 교체기록 없음:", len(wo_y - wo_chg), "건")
print("교체기록 있는데 Y가 아님:", len(wo_chg - wo_y), "건")
print("작업지시 1건당 교체 부품 행 수 최대:", partchg["WO_NO"].value_counts().max())

In [ ]:
# 정비 표는 44건을 유지함. 한 WO에서 부품이 여러 개여도 정비 행을 늘리지 않음.
# 교체 상세는 별도 표로 확인함. 여기서는 8개 WO에 9개 부품 행이 연결됨.
cal_chg = part_details.copy()

print('교체 없는 정비까지 남긴 정비:', len(maintenance_events), '건')
print('부품교체가 붙은 정비:', cal_chg['WO_NO'].nunique(), '건 → 교체 행:', len(cal_chg), '행')
display(cal_chg[['WO_NO', 'EQP_TAG', 'DETC_DT', 'MNT_TYPE', 'FAIL_MODE', 'PART_CD', 'QTY', 'CHG_RSN', 'REMARK']])



정비유형별 교체사유를 비교해 고장 후 교체와 예방 교체를 구분함.


In [ ]:
cross = pd.crosstab(cal_chg["MNT_TYPE"], cal_chg["CHG_RSN"]).reindex(MNT_ORDER)

display(cross)

정비 8건에서 부품교체 9행이 나온 것은 `WO-2024-00916`에서 2종을 교체했기 때문임. 손상 교체 4건 중 3건은 고장정비이며, `QTY`는 해당 작업의 부품 교체 수량(개)임.


## 4. 부품코드로 부품마스터 연결

`PART_CD`로 부품명, 분류, 적용설비, 표준수명(`STD_LIFE_H`, 시간), 조달기간(`LEAD_TIME_D`, 일)을 붙임.


In [ ]:
cal_part = part_details.copy()  # 앞 셀에서 PART_CD로 부품마스터까지 연결했음

print("부품마스터에서 못 찾은 PART_CD:", cal_part["PART_NM"].isna().sum(), "건")

show_cols = ["WO_NO", "DETC_DT", "MNT_TYPE", "PART_CD", "PART_NM", "PART_TYPE",
             "APPLY_EQP_TYPE", "STD_LIFE_H", "LEAD_TIME_D", "QTY", "CHG_RSN"]

display(cal_part[show_cols])

In [ ]:
# 부품별 요약: 교체 횟수(count)와 총 수량(sum)
display(cal_part.groupby(["PART_CD", "PART_NM"])["QTY"].agg(["count", "sum"]))

### 4-1. 부품별 교체 기록

점의 색은 정비유형, 옆 글자는 교체사유와 수량임. 실제 교체 시각이 없어 연결된 정비의 완료 시각에 표시함.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

for t in MNT_ORDER:
    sub = cal_part[cal_part["MNT_TYPE"] == t]
    ax.scatter(sub["END_DT"], sub["PART_NM"], color=MNT_COLORS[t], s=90, edgecolor="white", zorder=3, label=t)

for _, r in cal_part.iterrows():
    ax.annotate((str(r['CHG_RSN']) + ' ×' + str(r['QTY'])), (r["END_DT"], r["PART_NM"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8)

ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-25"))  # 12월 글자가 잘리지 않게 오른쪽 여유
ax.set_ylim(-0.6, 2.6)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 부품 교체 시점 (작업완료 END_DT 기준, 회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))

plt.savefig(os.path.join(FIG_DIR, "02_part_change_timeline.png"), bbox_inches="tight")

plt.show()

3종 부품이 각각 3차례 교체됐음. 6월 10일 정기수리 첫날에는 오일씰과 O링 세트를 함께 교체했음.


### 4-2. 교체 기록 사이의 간격

같은 `PART_CD`의 직전 정비 완료부터 이번 정비 완료까지 계산하며 첫 기록은 제외함. 부품 위치와 개체를 알 수 없어 실제 사용수명으로 해석하지 않음.


In [ ]:
cal_part = cal_part.sort_values("END_DT")

cal_part["동일코드_기록간격_h"] = cal_part.groupby("PART_CD")["END_DT"].diff().dt.total_seconds() / 3600

cal_part["표준수명대비_기록간격_%"] = cal_part["동일코드_기록간격_h"] / cal_part["STD_LIFE_H"] * 100

gaps = cal_part.dropna(subset=["동일코드_기록간격_h"])

display(gaps[["PART_CD", "PART_NM", "END_DT", "CHG_RSN", "동일코드_기록간격_h", "STD_LIFE_H", "표준수명대비_기록간격_%"]].round({"동일코드_기록간격_h": 1, "표준수명대비_기록간격_%": 1}))

In [ ]:
life = cal_part.groupby("PART_NM")["STD_LIFE_H"].first()  # 부품별 표준수명

fig, ax = plt.subplots(figsize=(11, 3))
ax.barh(life.index, life.values, color="#d9d9d9", height=0.5, label="표준수명 (STD_LIFE_H)")
ax.scatter(gaps["동일코드_기록간격_h"], gaps["PART_NM"], color="#2a78d6", s=70, zorder=3, label="동일 부품코드 기록 간격")
ax.set_xlabel("시간 (h)")
ax.set_title("표준수명(회색 막대) vs 동일 부품코드 기록 간격(파란 점)")
ax.legend(loc="lower right")

plt.savefig(os.path.join(FIG_DIR, "02_life_vs_interval.png"), bbox_inches="tight")

plt.show()

세 부품 모두 기록 간격이 표준수명의 40% 미만이지만, 표준수명은 설계 설정값이며 같은 개체의 사용시간도 알 수 없음. 간격을 계산한 6건은 손상 3건, 예방 2건, 수명도래 1건으로 조기 마모를 확정할 수 없음.

6월 10일 오일씰은 기록 간격이 표준수명의 약 23%인데 사유는 수명도래였음. 교체사유와 표준수명의 기준이 다를 수 있음.


### 4-3. 연속로 부품의 교체 여부

연속로 전용 11종 중 파랑은 2024년 CAL01에서 교체된 부품, 회색은 교체 기록이 없는 부품임.


In [ ]:
furnace = part[part["APPLY_EQP_TYPE"] == "연속로"].copy()

furnace["교체됨"] = furnace["PART_CD"].isin(cal_part["PART_CD"])

furnace = furnace.sort_values("STD_LIFE_H")

labels = furnace["PART_CD"] + " " + furnace["PART_NM"]
colors = []

for replaced in furnace["교체됨"]:
    if replaced:
        colors.append("#2a78d6")
    else:
        colors.append("#c3c2b7")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
axes[0].barh(labels, furnace["STD_LIFE_H"], color=colors)
axes[0].set_title("표준수명 STD_LIFE_H (시간)")
axes[1].barh(labels, furnace["LEAD_TIME_D"], color=colors)
axes[1].set_title("조달기간 LEAD_TIME_D (일)")
fig.suptitle("연속로 전용 부품 11개 (파랑 = 2024년 CAL01에서 교체, 회색 = 교체 기록 없음)", y=1.02)

plt.savefig(os.path.join(FIG_DIR, "02_furnace_parts.png"), bbox_inches="tight")

plt.show()

print("공통 + 연속로 부품 수:", part["APPLY_EQP_TYPE"].isin(["공통", "연속로"]).sum(), "개")
print("그중 CAL01에서 실제 교체된 부품:", sorted(cal_part["PART_CD"].unique()))

교체된 부품은 오일씰(`SEL-006`), O링 세트(`SEL-009`), 가스 분석계 셀(`INS-006`) 3종임. 히터, 열전대, CP 산소 프로브, 라디언트 튜브는 교체 기록이 없지만 열화가 없었다는 뜻은 아님.

오일씰과 O링을 교체한 고장정비의 고장모드는 전기(`ELE`)였음. 이 교체와 온도, 출력, CP 채널의 직접 연결 근거는 없음.

`CP-Z4`는 사전상 카본포텐셜이며 단위와 구체적 의미는 미확인임. 가스 분석계 셀과의 직접 연결도 확인되지 않았음.

조달기간은 가스 분석계 셀 31일, 퍼니스롤 85일, 라디언트 튜브 72일임. 예비품 계획에 참고할 수 있지만 모델이 반드시 그만큼 먼저 예측해야 한다는 뜻은 아님.


## 5. 정비와 부품교체 표 저장

정비는 작업당 1행, 부품은 교체 품목당 1행으로 저장함. 정비 결과와 교체 내용은 사후 확인 자료이며 과거에 이미 알려진 센서 입력으로 취급하지 않음.


In [ ]:
maintenance_events.to_csv(os.path.join(OUTPUT_DIR, '02_maintenance_events.csv'), index=False, encoding='utf-8-sig')
part_details.to_csv(os.path.join(OUTPUT_DIR, '02_part_details.csv'), index=False, encoding='utf-8-sig')

print('정비', len(maintenance_events), '건 / 교체 없는 정비', (~maintenance_events.HAS_PART_RECORD).sum(), '건 / 부품 상세', len(part_details), '행')